# Проверка fallback-ключа `ИНН + Номер договора` для `fix_comiss`

Цель:
- проверить, можно ли использовать ключ `ИНН + Номер договора эквайринга`, если нет `agr_id`;
- оценить качество матчинга (coverage / ambiguity);
- сравнить `fix_comiss` по двум методам:
  1) `by_agr_id` (gold),
  2) `by_inn_contract` (fallback).

In [ ]:
import re
from decimal import Decimal, InvalidOperation
import pandas as pd
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Параметры
excel_path = '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx'
month_start = '2026-02-01'
month_end = (pd.to_datetime(month_start) + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')

excel_inn_col = 'ИНН'
excel_contract_col = 'Номер договора'
excel_agr_col = 'ID договора'
excel_fix_col_candidates = ['Комиссия (Р в месяц)', 'Комиссия \n(₽ в месяц)']

eps_amount = 0.01

print('month_start =', month_start)
print('month_end =', month_end)

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()
with imp:
    imp.execute('invalidate metadata ods_alpha.scd1_agreements')
    imp.execute('invalidate metadata ods_alpha.scd1_companies')
    imp.execute('invalidate metadata ods.scd1_z_r2_ip_merchants')
    imp.execute('invalidate metadata ods.scd1_z_client')
    imp.execute('invalidate metadata ods.scd1_z_r2_tariff_tune')
    imp.execute('invalidate metadata ods.scd1_z_r2_tariff_fix')
print('Impala initialized')

In [ ]:
def normalize_numeric_str(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    s = s.replace(',', '.')
    if re.search(r"[eE][+-]?\d+$", s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r"\.0+$", '', s)
    s = re.sub(r"\D", '', s)
    if s == '':
        return None
    return s

def normalize_inn(v):
    s = normalize_numeric_str(v)
    if s is None:
        return None
    if len(s) in (10, 12):
        return s
    if len(s) == 9:
        return s.zfill(10)
    if len(s) == 11:
        return s.zfill(12)
    return None

def normalize_agr_id(v):
    return normalize_numeric_str(v)

def normalize_contract(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', ' ')
    s = re.sub(r"\s+", ' ', s)
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    return s.lower()

def normalize_num(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '')
    if s == '' or s.lower() in {'nan', 'none', 'null', '-'}:
        return None
    s = s.replace(',', '.')
    try:
        return float(Decimal(s))
    except Exception:
        return pd.to_numeric(s, errors='coerce')

def detect_excel_header_row(path, required_cols, scan_rows=30):
    raw = pd.read_excel(path, header=None)
    req = {str(c).strip() for c in required_cols}
    for i in range(min(scan_rows, len(raw))):
        vals = {str(x).strip() for x in raw.iloc[i].tolist()}
        if len(req & vals) >= 2:
            return i
    return 0

## 1) Excel-подготовка и базовые метрики уникальности

In [ ]:
header_row = detect_excel_header_row(excel_path, [excel_inn_col, excel_contract_col, excel_agr_col])
excel_raw = pd.read_excel(excel_path, header=header_row)
excel_raw.columns = [str(c).strip() for c in excel_raw.columns]

fix_col = next((c for c in excel_fix_col_candidates if c in excel_raw.columns), None)
if fix_col is None:
    raise ValueError(f'Fix commission column not found. candidates={excel_fix_col_candidates}')

excel_df = excel_raw[[excel_inn_col, excel_contract_col, excel_agr_col, fix_col]].copy()
excel_df.columns = ['inn_raw', 'contract_raw', 'agr_raw', 'fix_excel_raw']

excel_df['inn_norm'] = excel_df['inn_raw'].apply(normalize_inn)
excel_df['contract_norm'] = excel_df['contract_raw'].apply(normalize_contract)
excel_df['agr_id_norm'] = excel_df['agr_raw'].apply(normalize_agr_id)
excel_df['fix_excel'] = excel_df['fix_excel_raw'].apply(normalize_num)

excel_df['has_agr_id'] = excel_df['agr_id_norm'].notna().astype(int)

excel_key = excel_df.dropna(subset=['inn_norm', 'contract_norm']).copy()
excel_key_cnt = excel_key.groupby(['inn_norm', 'contract_norm'], as_index=False).size().rename(columns={'size': 'rows_per_key'})

stats_excel = pd.DataFrame([
    {'metric': 'rows_total', 'value': len(excel_df)},
    {'metric': 'rows_with_inn_contract', 'value': len(excel_key)},
    {'metric': 'rows_with_agr_id', 'value': int(excel_df['has_agr_id'].sum())},
    {'metric': 'unique_inn_contract_keys', 'value': excel_key[['inn_norm', 'contract_norm']].drop_duplicates().shape[0]},
    {'metric': 'duplicate_inn_contract_keys', 'value': int((excel_key_cnt['rows_per_key'] > 1).sum())},
])

display(stats_excel)
display(excel_key_cnt[excel_key_cnt['rows_per_key'] > 1].head(50))

## 2) Lake-ключи (`ИНН + Номер договора -> agr_id`) и качество матчинга

In [ ]:
sql_lake_contract_map = f"""
select
    case
      when length(regexp_replace(trim(c.c_inn), '[^0-9]', '')) = 9 then lpad(regexp_replace(trim(c.c_inn), '[^0-9]', ''), 10, '0')
      when length(regexp_replace(trim(c.c_inn), '[^0-9]', '')) = 11 then lpad(regexp_replace(trim(c.c_inn), '[^0-9]', ''), 12, '0')
      else regexp_replace(trim(c.c_inn), '[^0-9]', '')
    end as inn_norm,
    lower(trim(cast(a.c_agr_number as string))) as contract_norm,
    cast(a.abs_agr_id as string) as agr_id_norm,
    cast(a.n_agr as string) as n_agr
from ods_alpha.scd1_agreements a
join ods_alpha.scd1_companies c
  on c.n_cmp = a.n_cmp_client
where a.acq_class = 'SA'
  and cast(a.d_valid_from as date) <= cast('{month_start}' as date)
  and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
  and coalesce(a.ods_deleted_flg, '0') <> '1'
  and coalesce(c.ods_deleted_flg, '0') <> '1'
"""

with imp:
    lake_map = imp.fetch(sql_lake_contract_map)

lake_map = lake_map.dropna(subset=['inn_norm', 'contract_norm']).copy()
lake_map['agr_id_norm'] = lake_map['agr_id_norm'].apply(normalize_agr_id)

lake_key_card = (
    lake_map.groupby(['inn_norm', 'contract_norm'], as_index=False)
    .agg(
        lake_rows=('n_agr', 'count'),
        agr_candidates=('agr_id_norm', lambda s: int(s.dropna().nunique())),
        rows_without_agr=('agr_id_norm', lambda s: int(s.isna().sum()))
    )
)

stats_lake = pd.DataFrame([
    {'metric': 'lake_rows', 'value': len(lake_map)},
    {'metric': 'lake_unique_inn_contract', 'value': lake_key_card.shape[0]},
    {'metric': 'lake_keys_1_to_1', 'value': int((lake_key_card['agr_candidates'] == 1).sum())},
    {'metric': 'lake_keys_ambiguous_gt1', 'value': int((lake_key_card['agr_candidates'] > 1).sum())},
    {'metric': 'lake_keys_no_agr_id', 'value': int((lake_key_card['agr_candidates'] == 0).sum())},
])

display(stats_lake)
display(lake_key_card[(lake_key_card['agr_candidates'] > 1) | (lake_key_card['agr_candidates'] == 0)].head(100))

In [ ]:
# Покрытие fallback для Excel-строк без agr_id
excel_no_agr = excel_df[(excel_df['agr_id_norm'].isna()) & excel_df['inn_norm'].notna() & excel_df['contract_norm'].notna()].copy()
excel_no_agr_keys = excel_no_agr[['inn_norm', 'contract_norm']].drop_duplicates()

fallback_cov = (
    excel_no_agr_keys
    .merge(lake_key_card, on=['inn_norm', 'contract_norm'], how='left')
)

fallback_cov['agr_candidates'] = fallback_cov['agr_candidates'].fillna(0)
fallback_cov['rows_without_agr'] = fallback_cov['rows_without_agr'].fillna(0)

fallback_cov['match_status'] = fallback_cov['agr_candidates'].apply(
    lambda x: 'no_match' if x == 0 else ('matched_unique' if x == 1 else 'matched_ambiguous')
)

coverage_stats = (
    fallback_cov.groupby('match_status', as_index=False)
    .agg(keys_cnt=('inn_norm', 'count'))
    .sort_values('keys_cnt', ascending=False)
)

display(coverage_stats)
display(fallback_cov[fallback_cov['match_status'] != 'matched_unique'].head(100))

## 3) `fix_comiss` из R2 и A/B сравнение `by_agr_id` vs `by_inn_contract`

In [ ]:
sql_fix_r2 = """
select
    cast(m.id as string) as agr_id_norm,
    max(cast(tf.c_summa as double)) as fix_comiss,
    count(*) as r2_rows_cnt,
    count(distinct cast(tf.c_summa as string)) as fix_variants_cnt
from ods.scd1_z_r2_ip_merchants m
left join ods.scd1_z_r2_tariff_tune tt
  on tt.c_tariff_plan = m.c_tariff_plan
left join ods.scd1_z_r2_tariff_fix tf
  on tt.c_tariff = tf.id
where m.id is not null
group by cast(m.id as string)
"""

with imp:
    fix_r2 = imp.fetch(sql_fix_r2)

fix_r2['agr_id_norm'] = fix_r2['agr_id_norm'].apply(normalize_agr_id)
fix_r2 = fix_r2.dropna(subset=['agr_id_norm']).copy()

display(fix_r2.head(50))

In [ ]:
# Gold: by_agr_id
gold = excel_df[(excel_df['agr_id_norm'].notna()) & (excel_df['inn_norm'].notna()) & (excel_df['contract_norm'].notna())].copy()
gold = gold.merge(fix_r2[['agr_id_norm', 'fix_comiss', 'fix_variants_cnt']], on='agr_id_norm', how='left')
gold = gold.rename(columns={'fix_comiss': 'fix_by_agr_id', 'fix_variants_cnt': 'r2_fix_variants_by_agr'})

# Fallback: by_inn_contract -> lake agr -> r2 fix
lake_unique = lake_key_card[lake_key_card['agr_candidates'] == 1][['inn_norm', 'contract_norm']].copy()
lake_unique = lake_unique.merge(
    lake_map[['inn_norm', 'contract_norm', 'agr_id_norm']].drop_duplicates(),
    on=['inn_norm', 'contract_norm'], how='left'
)

gold_fb = (
    gold[['inn_norm', 'contract_norm', 'agr_id_norm', 'fix_excel', 'fix_by_agr_id']]
    .merge(lake_unique, on=['inn_norm', 'contract_norm'], how='left', suffixes=('', '_fb'))
)

gold_fb = gold_fb.merge(
    fix_r2[['agr_id_norm', 'fix_comiss']],
    left_on='agr_id_norm_fb', right_on='agr_id_norm', how='left', suffixes=('', '_fb_fix')
)

gold_fb = gold_fb.rename(columns={'fix_comiss': 'fix_by_inn_contract'})
gold_fb['fb_matches_same_agr'] = (gold_fb['agr_id_norm'] == gold_fb['agr_id_norm_fb'])
gold_fb['ab_diff'] = gold_fb['fix_by_inn_contract'] - gold_fb['fix_by_agr_id']
gold_fb['ab_match'] = gold_fb['ab_diff'].apply(lambda x: abs(x) <= eps_amount if pd.notna(x) else None)

ab_summary = pd.DataFrame([
    {'metric': 'gold_rows', 'value': len(gold_fb)},
    {'metric': 'fallback_has_unique_key', 'value': int(gold_fb['agr_id_norm_fb'].notna().sum())},
    {'metric': 'fallback_same_agr_as_gold', 'value': int(gold_fb['fb_matches_same_agr'].sum())},
    {'metric': 'ab_compared_rows', 'value': int(gold_fb['ab_match'].notna().sum())},
    {'metric': 'ab_match_rows', 'value': int((gold_fb['ab_match'] == True).sum())},
    {'metric': 'ab_mismatch_rows', 'value': int((gold_fb['ab_match'] == False).sum())},
])

display(ab_summary)
display(gold_fb[gold_fb['ab_match'] == False].head(100))

## 4) Рекомендация по применению fallback

In [ ]:
recommendation = pd.DataFrame([
    {
        'rule': '1) by_agr_id',
        'description': 'Основной путь. Использовать, если agr_id есть и fix_variants_cnt <= 1.'
    },
    {
        'rule': '2) by_inn_contract',
        'description': 'Fallback только для ключей с unique match (agr_candidates = 1).'
    },
    {
        'rule': '3) ambiguous/no_match',
        'description': 'Ставить NULL и флаг качества, не подставлять fix_comiss принудительно.'
    },
])

display(recommendation)